# Mondrian conformal HDF5 demo

This notebook runs Mondrian conformal analysis using saved CNN predictions, labels, and embeddings.

It supports two modes:

- real calibration/test mode, using `pred_cal`, `y_cal`, `emb_cal`, `pred_test`, `y_test`, and `emb_test`
- debug mode, where validation predictions are split into pseudo-calibration and pseudo-test subsets

Debug mode is useful for testing the pipeline, but must not be reported as final conformal performance.

In [ ]:
from pathlib import Path
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import iqr

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name != "cbc_pe" and (PROJECT_ROOT / "cbc_pe").exists():
    PROJECT_ROOT = PROJECT_ROOT / "cbc_pe"

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_ROOT = Path("/scratch/vserrano/cbc_pe_data")
DATA_RESULTS = DATA_ROOT / "results"

dataset_id = "bbh_processed_4s_seobnrv4opt_snr10-25_n100_000"
RESULTS_DIR = DATA_RESULTS / dataset_id

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("RESULTS_DIR:", RESULTS_DIR)
print("RESULTS_DIR exists:", RESULTS_DIR.exists())

## Select prediction file

For the current 100k architecture-search setup, calibration and test sets are not available. Use debug mode only for pipeline development.

For final reporting, use a prediction file produced from a 70/10/10/10 or similar train/validation/calibration/test split.

In [ ]:
MODEL_ID = "M00_baseline_emb64_seed123"

prediction_files = {
    "M00_baseline_emb64_seed123": RESULTS_DIR / (
        "bbh_processed_4s_seobnrv4opt_snr10-25_n100_000"
        "_SimpleCNN_Baseline_M00_simple_emb64_mse_MSELoss_seed123"
        "_train_val_predictions_embeddings.npz"
    ),
    "M00_baseline_emb64_seed124": RESULTS_DIR / (
        "bbh_processed_4s_seobnrv4opt_snr10-25_n100_000"
        "_SimpleCNN_Baseline_M00_simple_emb64_mse_MSELoss_seed124"
        "_train_val_predictions_embeddings.npz"
    ),
    "M04_pooldeep_emb128_pool4": RESULTS_DIR / (
        "bbh_processed_4s_seobnrv4opt_snr10-25_n100_000"
        "_SimpleCNN_PoolDeep_M04_emb128_pool4_deephead_MSELoss_seed123"
        "_train_val_predictions_embeddings.npz"
    ),
}

prediction_file = prediction_files[MODEL_ID]

assert prediction_file.exists(), prediction_file

print("MODEL_ID:", MODEL_ID)
print("prediction_file:", prediction_file)

data = np.load(prediction_file, allow_pickle=True)
print(data.files)

## Build calibration and test arrays

In [ ]:
def load_conformal_arrays(
    data,
    debug_split_val=False,
    debug_cal_fraction=0.5,
    debug_seed=123,
):
    """
    Load arrays needed for Mondrian/conformal.

    Preferred real mode:
        pred_cal, y_cal, emb_cal
        pred_test, y_test, emb_test

    Debug mode:
        If real cal/test are not available, split val into pseudo_cal/pseudo_test.
        This is only for debugging the notebook, not for final reporting.
    """

    files = set(data.files)

    has_real_cal_test = {
        "pred_cal",
        "y_cal",
        "emb_cal",
        "pred_test",
        "y_test",
        "emb_test",
    }.issubset(files)

    if has_real_cal_test:
        print("Using real cal/test arrays.")

        pred_cal = data["pred_cal"]
        y_cal = data["y_cal"]
        emb_cal = data["emb_cal"]

        pred_test = data["pred_test"]
        y_test = data["y_test"]
        emb_test = data["emb_test"]

        idx_cal = data["idx_cal"] if "idx_cal" in files else None
        idx_test = data["idx_test"] if "idx_test" in files else None

        mode = "real_cal_test"

    else:
        if not debug_split_val:
            raise KeyError(
                "No real cal/test arrays found in prediction file. "
                "Expected pred_cal/y_cal/emb_cal and pred_test/y_test/emb_test. "
                "For notebook debugging only, set debug_split_val=True."
            )

        required_val = {"pred_val", "y_val", "emb_val"}

        if not required_val.issubset(files):
            raise KeyError(
                "Cannot create debug split because pred_val/y_val/emb_val are missing."
            )

        print("WARNING: using validation split as pseudo cal/test.")
        print("This is only for debugging. Do not report these results as final.")

        pred_val = data["pred_val"]
        y_val = data["y_val"]
        emb_val = data["emb_val"]

        n_val = pred_val.shape[0]
        rng = np.random.default_rng(debug_seed)
        perm = rng.permutation(n_val)

        n_cal = int(debug_cal_fraction * n_val)

        cal_local = perm[:n_cal]
        test_local = perm[n_cal:]

        pred_cal = pred_val[cal_local]
        y_cal = y_val[cal_local]
        emb_cal = emb_val[cal_local]

        pred_test = pred_val[test_local]
        y_test = y_val[test_local]
        emb_test = emb_val[test_local]

        if "idx_val" in files:
            idx_val = data["idx_val"]
            idx_cal = idx_val[cal_local]
            idx_test = idx_val[test_local]
        else:
            idx_cal = cal_local
            idx_test = test_local

        mode = "debug_val_split"

    y_mean = data["y_mean"]
    y_std = data["y_std"]

    if "label_names" in files:
        label_names = data["label_names"].tolist()
    else:
        label_names = ["chirp_mass", "total_mass", "chi_eff"]

    return {
        "mode": mode,
        "pred_cal": pred_cal,
        "y_cal": y_cal,
        "emb_cal": emb_cal,
        "pred_test": pred_test,
        "y_test": y_test,
        "emb_test": emb_test,
        "idx_cal": idx_cal,
        "idx_test": idx_test,
        "y_mean": y_mean,
        "y_std": y_std,
        "label_names": label_names,
    }

In [ ]:
DEBUG_SPLIT_VAL = True  # Debug only. Do not report as final conformal performance.

arrays = load_conformal_arrays(
    data,
    debug_split_val=DEBUG_SPLIT_VAL,
    debug_cal_fraction=0.5,
    debug_seed=123,
)

mode = arrays["mode"]

pred_cal = arrays["pred_cal"]
y_cal = arrays["y_cal"]
emb_cal = arrays["emb_cal"]

pred_test = arrays["pred_test"]
y_test = arrays["y_test"]
emb_test = arrays["emb_test"]

y_mean = arrays["y_mean"]
y_std = arrays["y_std"]
label_names = arrays["label_names"]

print("mode:", mode)
print("pred_cal:", pred_cal.shape)
print("pred_test:", pred_test.shape)
print("labels:", label_names)

if mode != "real_cal_test":
    print("WARNING: running in debug pseudo-cal/test mode. Do not report as final conformal performance.")

In [ ]:
mode = arrays["mode"]

pred_cal = arrays["pred_cal"]
y_cal = arrays["y_cal"]
emb_cal = arrays["emb_cal"]

pred_test = arrays["pred_test"]
y_test = arrays["y_test"]
emb_test = arrays["emb_test"]

idx_cal = arrays["idx_cal"]
idx_test = arrays["idx_test"]

y_mean = arrays["y_mean"]
y_std = arrays["y_std"]
label_names = arrays["label_names"]

print("mode:", mode)
print("pred_cal:", pred_cal.shape)
print("y_cal:", y_cal.shape)
print("emb_cal:", emb_cal.shape)
print("pred_test:", pred_test.shape)
print("y_test:", y_test.shape)
print("emb_test:", emb_test.shape)
print("y_mean:", y_mean)
print("y_std:", y_std)
print("label_names:", label_names)

In [ ]:
#Sanity Checks
assert pred_cal.ndim == 2
assert pred_test.ndim == 2
assert y_cal.ndim == 2
assert y_test.ndim == 2

assert pred_cal.shape == y_cal.shape
assert pred_test.shape == y_test.shape

assert pred_cal.shape[1] == len(label_names)
assert pred_test.shape[1] == len(label_names)

assert emb_cal.ndim == 2
assert emb_test.ndim == 2

assert emb_cal.shape[0] == pred_cal.shape[0]
assert emb_test.shape[0] == pred_test.shape[0]

assert y_mean.shape[0] == len(label_names)
assert y_std.shape[0] == len(label_names)
assert np.all(y_std > 0)

for name, arr in {
    "pred_cal": pred_cal,
    "y_cal": y_cal,
    "pred_test": pred_test,
    "y_test": y_test,
    "emb_cal": emb_cal,
    "emb_test": emb_test,
    "y_mean": y_mean,
    "y_std": y_std,
}.items():
    assert np.all(np.isfinite(arr)), f"{name} contains NaN or inf"

if mode != "real_cal_test":
    print("WARNING: notebook is running in debug mode:", mode)
    print("Do not use these results as final conformal/Mondrian results.")

print("All sanity checks passed.")

In [ ]:
def inverse_standardize(y_std_values, y_mean, y_std):
    return y_std_values * y_std + y_mean


y_cal_phys = inverse_standardize(y_cal, y_mean, y_std)
y_test_phys = inverse_standardize(y_test, y_mean, y_std)

pred_cal_phys = inverse_standardize(pred_cal, y_mean, y_std)
pred_test_phys = inverse_standardize(pred_test, y_mean, y_std)

label_ranges_phys = {
    label: np.max(y_test_phys[:, j]) - np.min(y_test_phys[:, j])
    for j, label in enumerate(label_names)
}

label_ranges_phys

## Start Mondrian

In [ ]:
confidence_level = 0.90

if mode == "real_cal_test":
    n_bins_grid = [4, 6, 8, 12, 16, 24, 32, 48]
else:
    n_bins_grid = [4, 6, 8, 12, 16, 24]

taxonomy_modes = ["prediction", "difficulty"]
interval_modes = ["symmetric", "asymmetric"]

n_neighbors = 5
min_samples_per_bin = 20 if mode == "real_cal_test" else 10

rows = []
all_results = {}

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("/afs/ciemat.es/user/v/vserrano/Desktop/gw/Gravitational-Waves-Lab/cbc_pe")  # ajusta si tu repo está en otra ruta

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(PROJECT_ROOT)

In [ ]:
from src.conformal.pipeline import run_mondrian_regression

for taxonomy_mode in taxonomy_modes:
    for interval_mode in interval_modes:
        for n_bins in n_bins_grid:

            kwargs = dict(
                pred_cal=pred_cal,
                pred_test=pred_test,
                y_cal=y_cal,
                y_test=y_test,
                n_bins=n_bins,
                confidence_level=confidence_level,
                apply_jitter=True,
                interval_mode=interval_mode,
                taxonomy_mode=taxonomy_mode,
                min_samples_per_bin=min_samples_per_bin,
                tolerance_sigmas=(1, 2, 3),
            )

            if taxonomy_mode == "difficulty":
                kwargs.update(
                    cal_embedding=emb_cal,
                    target_embedding=emb_test,
                    n_neighbors=n_neighbors,
                )

            result = run_mondrian_regression(**kwargs)
            all_results[(taxonomy_mode, interval_mode, n_bins)] = result

            metrics = result.metrics

            widths_std = result.upper - result.lower
            widths_phys = widths_std * y_std

            for j, label in enumerate(label_names):
                n_bad_bins_p005 = int(
                    np.nansum(metrics["bin_undercoverage_pvalue"][:, j] < 0.05)
                )

                row = {
                    "mode": mode,
                    "taxonomy_mode": taxonomy_mode,
                    "interval_mode": interval_mode,
                    "n_bins": n_bins,
                    "label": label,
                    "label_index": j,

                    "n_samples": int(metrics["n_samples_per_label"][j]),
                    "covered_count": int(metrics["covered_count_global"][j]),

                    "global_coverage": metrics["global_coverage"][j],
                    "miscoverage": metrics["miscoverage"][j],
                    "global_coverage_gap": metrics["global_coverage_gap"][j],
                    "global_undercoverage_pvalue": metrics["global_undercoverage_pvalue"][j],

                    "global_mean_width_std": metrics["global_mean_width"][j],
                    "global_median_width_std": metrics["global_median_width"][j],

                    "global_mean_width_phys": np.mean(widths_phys[:, j]),
                    "global_median_width_phys": np.median(widths_phys[:, j]),
                    "global_iqr_width_phys": iqr(widths_phys[:, j]),

                    "normalized_median_width_range": (
                        np.median(widths_phys[:, j]) / label_ranges_phys[label]
                    ),

                    "min_coverage_per_bin": metrics["min_coverage_per_label"][j],
                    "max_undercoverage_gap": metrics["max_undercoverage_gap"][j],

                    "min_count_per_bin": int(np.nanmin(metrics["counts_per_bin"][:, j])),
                    "max_count_per_bin": int(np.nanmax(metrics["counts_per_bin"][:, j])),

                    "n_bad_bins_p005": n_bad_bins_p005,
                    "bad_bin_fraction": n_bad_bins_p005 / n_bins,

                    "global_lower_miss_rate": metrics["global_lower_miss_rate"][j],
                    "global_upper_miss_rate": metrics["global_upper_miss_rate"][j],
                    "global_tail_miss_imbalance": metrics["global_tail_miss_imbalance"][j],
                }

                global_tol = metrics["global_tolerance_normal"]

                for k in [1, 2, 3]:
                    low = global_tol[f"{k}sigma_low"][j]
                    high = global_tol[f"{k}sigma_high"][j]
                    width = global_tol[f"{k}sigma_width"][j]

                    row[f"global_tol_{k}sigma_low"] = low
                    row[f"global_tol_{k}sigma_high"] = high
                    row[f"global_tol_{k}sigma_width"] = width
                    row[f"global_within_{k}sigma"] = bool(low <= metrics["global_coverage"][j] <= high)

                rows.append(row)

summary_df = pd.DataFrame(rows)
summary_df

## Save the results

In [ ]:
mondrian_results_dir = RESULTS_DIR / "mondrian"
mondrian_results_dir.mkdir(parents=True, exist_ok=True)

prediction_stem = prediction_file.stem

summary_path = mondrian_results_dir / f"{prediction_stem}_mondrian_summary_{mode}.csv"

summary_df.to_csv(summary_path, index=False)

print("Saved:", summary_path)

## Ranking Configurations

### Local coverage diagnostics: bin-wise 2σ / 3σ criteria

The global coverage can look good while some Mondrian bins are locally undercovered.

This cell adds explicit bin-level diagnostics based on the binomial standard error:

$$
\sigma_b = \sqrt{\frac{CL(1-CL)}{n_b}}
$$

where `CL` is the target coverage level, e.g. 0.90.


In [ ]:

# ============================================================
# Add local bin-wise coverage diagnostics to summary_df
# ============================================================

def compute_local_bin_diagnostics(
    all_results,
    summary_df,
    label_names,
    confidence_level=0.90,
):
    """
    Add bin-wise local coverage diagnostics to summary_df.

    We treat undercoverage as the main failure mode:
        coverage_bin < confidence_level - k * sigma_bin

    We also compute two-sided outside-ksigma counts, but use undercoverage
    as the main selection criterion because overcoverage is conservative.
    """
    rows = []

    for (taxonomy_mode, interval_mode, n_bins), result in all_results.items():
        metrics = result.metrics

        coverage_per_bin = metrics["coverage_per_bin"]       # (n_bins, n_labels)
        counts_per_bin = metrics["counts_per_bin"]           # (n_bins, n_labels)

        for j, label in enumerate(label_names):
            cov = coverage_per_bin[:, j].astype(float)
            counts = counts_per_bin[:, j].astype(float)

            valid = np.isfinite(cov) & np.isfinite(counts) & (counts > 0)

            sigma = np.full_like(cov, np.nan, dtype=float)
            sigma[valid] = np.sqrt(confidence_level * (1.0 - confidence_level) / counts[valid])

            row = {
                "taxonomy_mode": taxonomy_mode,
                "interval_mode": interval_mode,
                "n_bins": n_bins,
                "label": label,
                "label_index": j,

                "mean_bin_sigma": float(np.nanmean(sigma)),
                "max_bin_sigma": float(np.nanmax(sigma)),
                "min_bin_coverage": float(np.nanmin(cov)),
                "max_bin_coverage": float(np.nanmax(cov)),
                "bin_coverage_std": float(np.nanstd(cov)),
            }

            for k in [1, 2, 3]:
                low = confidence_level - k * sigma
                high = confidence_level + k * sigma

                under = valid & (cov < low)
                over = valid & (cov > high)
                outside = under | over

                n_under = int(np.sum(under))
                n_over = int(np.sum(over))
                n_outside = int(np.sum(outside))

                row[f"n_under_bins_{k}sigma"] = n_under
                row[f"n_over_bins_{k}sigma"] = n_over
                row[f"n_outside_bins_{k}sigma"] = n_outside

                row[f"under_bin_fraction_{k}sigma"] = n_under / n_bins
                row[f"outside_bin_fraction_{k}sigma"] = n_outside / n_bins

                # Severity beyond the statistical lower tolerance.
                if np.any(under):
                    row[f"max_undercoverage_beyond_{k}sigma"] = float(np.nanmax(low[under] - cov[under]))
                else:
                    row[f"max_undercoverage_beyond_{k}sigma"] = 0.0

                row[f"all_bins_within_{k}sigma"] = bool(n_outside == 0)
                row[f"all_bins_not_under_{k}sigma"] = bool(n_under == 0)

            rows.append(row)

    diagnostics_df = pd.DataFrame(rows)

    merge_keys = ["taxonomy_mode", "interval_mode", "n_bins", "label", "label_index"]

    out = summary_df.merge(
        diagnostics_df,
        on=merge_keys,
        how="left",
        validate="one_to_one",
    )

    return out


summary_df = compute_local_bin_diagnostics(
    all_results=all_results,
    summary_df=summary_df,
    label_names=label_names,
    confidence_level=confidence_level,
)

local_diag_cols = [
    "label",
    "taxonomy_mode",
    "interval_mode",
    "n_bins",
    "global_coverage",
    "global_within_2sigma",
    "min_bin_coverage",
    "max_undercoverage_gap",
    "n_under_bins_2sigma",
    "under_bin_fraction_2sigma",
    "max_undercoverage_beyond_2sigma",
    "n_under_bins_3sigma",
    "under_bin_fraction_3sigma",
    "max_undercoverage_beyond_3sigma",
    "global_median_width_phys",
    "min_count_per_bin",
]

summary_df[local_diag_cols].sort_values(
    ["label", "under_bin_fraction_2sigma", "global_median_width_phys", "n_bins"],
    ascending=[True, True, True, False],
).head(20)


### Ranking with global and local validity constraints

Selection is hierarchical:

1. global coverage must be statistically valid;
2. local bin undercoverage must be controlled using 2σ criteria;
3. each bin must have enough test samples;
4. among valid configurations, choose small width;
5. if widths are effectively tied, prefer more bins;
6. if local 2σ validity is too strict for a label, inspect a relaxed 3σ table separately.


In [ ]:

# ============================================================
# Ranking with local coverage constraints
# ============================================================

# --- Main thresholds ---
GLOBAL_SIGMA_LEVEL = 2
LOCAL_SIGMA_LEVEL = 2

# For real cal/test with 30k test samples, 200 is a more meaningful lower bound than 20.
# With quantile bins this will usually be much larger, but keep the guard explicit.
min_count_threshold = 200 if mode == "real_cal_test" else 50

# Allow at most 10% of bins to be locally undercovered beyond the chosen sigma.
# For n_bins=32 this allows at most 3 bins; for n_bins=4 it allows 0 bins.
max_under_bin_fraction = 0.10

# Hard severity cutoff relative to nominal CL.
# This avoids choosing configs with very bad local failures even if they pass sigma tests.
max_nominal_undercoverage_gap = 0.05

# Prefer more bins only when the width is within this relative tolerance
# of the best locally-valid width for the same label.
width_tie_relative_tolerance = 0.02  # 2%

global_col = f"global_within_{GLOBAL_SIGMA_LEVEL}sigma"
under_frac_col = f"under_bin_fraction_{LOCAL_SIGMA_LEVEL}sigma"
n_under_col = f"n_under_bins_{LOCAL_SIGMA_LEVEL}sigma"
beyond_col = f"max_undercoverage_beyond_{LOCAL_SIGMA_LEVEL}sigma"

candidate_df = summary_df[
    (summary_df[global_col]) &
    (summary_df["min_count_per_bin"] >= min_count_threshold) &
    (summary_df[under_frac_col] <= max_under_bin_fraction) &
    (summary_df["max_undercoverage_gap"] <= max_nominal_undercoverage_gap)
].copy()

print("Mode:", mode)
print("Total configs by label:")
print(summary_df.groupby("label").size())
print()
print("Candidate configs after global/local filters:")
print(candidate_df.groupby("label").size())

candidate_df["best_width_for_label"] = candidate_df.groupby("label")["global_median_width_phys"].transform("min")
candidate_df["rel_width_excess"] = (
    candidate_df["global_median_width_phys"] / candidate_df["best_width_for_label"] - 1.0
)
candidate_df["within_width_tie"] = candidate_df["rel_width_excess"] <= width_tie_relative_tolerance

# Main ranking:
# - Prefer configs within 2% of the best valid width.
# - Among tied widths, prefer more bins.
# - Then prefer less undercoverage and smaller width.
ranking_df = candidate_df.sort_values(
    [
        "label",
        "within_width_tie",
        "n_bins",
        under_frac_col,
        "max_undercoverage_gap",
        "global_median_width_phys",
        "global_tail_miss_imbalance",
    ],
    ascending=[
        True,
        False,
        False,
        True,
        True,
        True,
        True,
    ],
).reset_index(drop=True)

display_cols = [
    "label",
    "taxonomy_mode",
    "interval_mode",
    "n_bins",
    "global_coverage",
    global_col,
    "min_bin_coverage",
    "max_undercoverage_gap",
    n_under_col,
    under_frac_col,
    beyond_col,
    "global_median_width_phys",
    "rel_width_excess",
    "within_width_tie",
    "global_tail_miss_imbalance",
    "min_count_per_bin",
]

ranking_df[display_cols].head(30)


### We see the best k configs per label

In [ ]:
# best k configs for each label under the local-validity ranking
top_k = 5

top_by_label = (
    ranking_df
    .groupby("label", group_keys=False)
    .head(top_k)
)

top_cols = [
    "label",
    "taxonomy_mode",
    "interval_mode",
    "n_bins",
    "global_coverage",
    "min_bin_coverage",
    "max_undercoverage_gap",
    n_under_col,
    under_frac_col,
    "global_median_width_std",
    "global_median_width_phys",
    "rel_width_excess",
    "within_width_tie",
    "global_tail_miss_imbalance",
    "min_count_per_bin",
]

top_by_label[top_cols]

### Relaxed 3σ diagnostic table

Use this only as a diagnostic.  
The main table above uses 2σ local undercoverage. If a label has no acceptable 2σ candidates, inspect this relaxed 3σ view and report the relaxation explicitly.


In [ ]:

# ============================================================
# Relaxed local criterion: 3 sigma
# ============================================================

RELAXED_LOCAL_SIGMA_LEVEL = 3
relaxed_under_frac_col = f"under_bin_fraction_{RELAXED_LOCAL_SIGMA_LEVEL}sigma"
relaxed_n_under_col = f"n_under_bins_{RELAXED_LOCAL_SIGMA_LEVEL}sigma"
relaxed_beyond_col = f"max_undercoverage_beyond_{RELAXED_LOCAL_SIGMA_LEVEL}sigma"

relaxed_candidate_df = summary_df[
    (summary_df[global_col]) &
    (summary_df["min_count_per_bin"] >= min_count_threshold) &
    (summary_df[relaxed_under_frac_col] <= max_under_bin_fraction) &
    (summary_df["max_undercoverage_gap"] <= max_nominal_undercoverage_gap)
].copy()

print("Relaxed candidate configs after 3sigma local filter:")
print(relaxed_candidate_df.groupby("label").size())

relaxed_candidate_df["best_width_for_label"] = relaxed_candidate_df.groupby("label")["global_median_width_phys"].transform("min")
relaxed_candidate_df["rel_width_excess"] = (
    relaxed_candidate_df["global_median_width_phys"] / relaxed_candidate_df["best_width_for_label"] - 1.0
)
relaxed_candidate_df["within_width_tie"] = relaxed_candidate_df["rel_width_excess"] <= width_tie_relative_tolerance

relaxed_ranking_df = relaxed_candidate_df.sort_values(
    [
        "label",
        "within_width_tie",
        "n_bins",
        relaxed_under_frac_col,
        "max_undercoverage_gap",
        "global_median_width_phys",
        "global_tail_miss_imbalance",
    ],
    ascending=[True, False, False, True, True, True, True],
).reset_index(drop=True)

relaxed_cols = [
    "label",
    "taxonomy_mode",
    "interval_mode",
    "n_bins",
    "global_coverage",
    global_col,
    "min_bin_coverage",
    "max_undercoverage_gap",
    relaxed_n_under_col,
    relaxed_under_frac_col,
    relaxed_beyond_col,
    "global_median_width_phys",
    "rel_width_excess",
    "within_width_tie",
    "global_tail_miss_imbalance",
    "min_count_per_bin",
]

relaxed_ranking_df[relaxed_cols].head(30)


## Checking a configuration

In [ ]:
def inspect_configuration(
    all_results,
    key,
    label_idx,
    label_names,
):
    result = all_results[key]
    metrics = result.metrics
    label = label_names[label_idx]

    print("Configuration:", key)
    print("Label:", label)

    print("\nCounts per bin:")
    print(metrics["counts_per_bin"][:, label_idx])

    print("\nCoverage per bin:")
    print(np.round(metrics["coverage_per_bin"][:, label_idx], 3))

    print("\nCovered count per bin:")
    print(metrics["covered_count_per_bin"][:, label_idx])

    print("\nBin undercoverage p-values:")
    print(np.round(metrics["bin_undercoverage_pvalue"][:, label_idx], 4))

    print("\nLower miss rate per bin:")
    print(np.round(metrics["lower_miss_rate_per_bin"][:, label_idx], 3))

    print("\nUpper miss rate per bin:")
    print(np.round(metrics["upper_miss_rate_per_bin"][:, label_idx], 3))

    print("\nInterval offsets per bin:")
    print(np.round(result.intervals[label_idx], 4))

    print("\nCalibrator bin counts:")
    print(result.calibrator.bin_counts_[label_idx])

    if hasattr(result.calibrator, "quantile_indices_"):
        print("\nQuantile indices:")
        print(result.calibrator.quantile_indices_[label_idx])

In [ ]:
key = ("difficulty", "asymmetric", 12)
inspect_configuration(
    all_results=all_results,
    key=key,
    label_idx=0,
    label_names=label_names,
)

## Plots

In [ ]:
plot_labels = {
    "chirp_mass": r"$\mathcal{M}$",
    "total_mass": r"$M_{\mathrm{tot}}$",
    "chi_eff": r"$\chi_{\mathrm{eff}}$",
}

plot_colors = {
    "symmetric": "teal",
    "asymmetric": "dodgerblue",
}


def plot_global_coverage_vs_bins(
    summary_df,
    label,
    taxonomy_modes,
    interval_modes,
    confidence_level,
    plot_labels,
    plot_colors=None,
):
    df_label = summary_df[summary_df["label"] == label]

    fig, axes = plt.subplots(
        1,
        len(taxonomy_modes),
        figsize=(14, 5),
        sharey=True,
        constrained_layout=True,
    )

    axes = np.atleast_1d(axes)

    for ax, taxonomy_mode in zip(axes, taxonomy_modes):
        df_tax = df_label[df_label["taxonomy_mode"] == taxonomy_mode]

        if df_tax.empty:
            ax.set_title(f"{taxonomy_mode} (no data)")
            continue

        # Global tolerance is constant for this label.
        low_1 = df_tax["global_tol_1sigma_low"].iloc[0]
        high_1 = df_tax["global_tol_1sigma_high"].iloc[0]
        low_2 = df_tax["global_tol_2sigma_low"].iloc[0]
        high_2 = df_tax["global_tol_2sigma_high"].iloc[0]
        low_3 = df_tax["global_tol_3sigma_low"].iloc[0]
        high_3 = df_tax["global_tol_3sigma_high"].iloc[0]

        ax.axhspan(low_1, high_1, color="red", alpha=0.60, label=r"$1\sigma$")
        ax.axhspan(low_2, high_2, color="red", alpha=0.40, label=r"$2\sigma$")
        ax.axhspan(low_3, high_3, color="red", alpha=0.20, label=r"$3\sigma$")

        ax.axhline(
            confidence_level,
            color="black",
            linestyle="--",
            linewidth=1,
            label=rf"C.L. = {int(confidence_level * 100)}%",
        )

        for interval_mode in interval_modes:
            df_mode = (
                df_tax[df_tax["interval_mode"] == interval_mode]
                .sort_values("n_bins")
            )

            color = None if plot_colors is None else plot_colors.get(interval_mode)

            ax.plot(
                df_mode["n_bins"],
                df_mode["global_coverage"],
                marker="o",
                label=interval_mode,
                color=color,
            )

        ax.set_xticks(sorted(df_tax["n_bins"].unique()))
        ax.set_xlabel(r"$n_{\mathrm{bins}}$", fontsize=13)
        ax.set_title(taxonomy_mode)
        ax.grid(alpha=0.25)

    axes[0].set_ylabel("Global coverage", fontsize=13)

    fig.suptitle(f"Coverage vs bins ({plot_labels[label]})", fontsize=16)

    handles, labels_legend = axes[0].get_legend_handles_labels()
    fig.legend(
        handles,
        labels_legend,
        loc="upper left",
        bbox_to_anchor=(0.82, 1.08),
        ncol=2,
    )

    plt.show()

In [ ]:
for label in label_names:
    plot_global_coverage_vs_bins(
        summary_df=summary_df,
        label=label,
        taxonomy_modes=taxonomy_modes,
        interval_modes=interval_modes,
        confidence_level=confidence_level,
        plot_labels=plot_labels,
        plot_colors=plot_colors,
    )

In [ ]:
def plot_width_vs_bins(
    summary_df,
    label,
    taxonomy_modes,
    interval_modes,
    plot_labels,
    width_column="global_median_width_std",
    plot_colors=None,
):
    df_label = summary_df[summary_df["label"] == label]

    fig, axes = plt.subplots(
        1,
        len(taxonomy_modes),
        figsize=(14, 5),
        sharey=True,
        constrained_layout=True,
    )

    axes = np.atleast_1d(axes)

    for ax, taxonomy_mode in zip(axes, taxonomy_modes):
        df_tax = df_label[df_label["taxonomy_mode"] == taxonomy_mode]

        for interval_mode in interval_modes:
            df_mode = (
                df_tax[df_tax["interval_mode"] == interval_mode]
                .sort_values("n_bins")
            )

            color = None if plot_colors is None else plot_colors.get(interval_mode)

            ax.plot(
                df_mode["n_bins"],
                df_mode[width_column],
                marker="o",
                label=interval_mode,
                color=color,
            )

        ax.set_title(taxonomy_mode)
        ax.set_xlabel(r"$n_{\mathrm{bins}}$", fontsize=13)
        ax.set_xticks(sorted(df_tax["n_bins"].unique()))
        ax.grid(alpha=0.25)

    axes[0].set_ylabel(width_column, fontsize=13)
    fig.suptitle(f"Interval width vs bins ({plot_labels[label]})", fontsize=16)

    handles, labels_legend = axes[0].get_legend_handles_labels()
    fig.legend(
        handles,
        labels_legend,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.08),
        ncol=2,
    )

    plt.show()

In [ ]:
for label in label_names:
    plot_width_vs_bins(
        summary_df=summary_df,
        label=label,
        taxonomy_modes=taxonomy_modes,
        interval_modes=interval_modes,
        plot_labels=plot_labels,
        width_column="global_median_width_phys",
        plot_colors=plot_colors,
    )

In [ ]:
def plot_coverage_width_tradeoff(
    summary_df,
    label,
    taxonomy_modes,
    interval_modes,
    confidence_level,
    plot_labels,
    width_column="global_median_width_std",
):
    df_label = summary_df[summary_df["label"] == label]

    scatter_color = {

    }

    plt.figure(figsize=(16, 10))

    for taxonomy_mode in taxonomy_modes:
        for interval_mode in interval_modes:
            df_mode = df_label[
                (df_label["taxonomy_mode"] == taxonomy_mode) &
                (df_label["interval_mode"] == interval_mode)
            ]

            plt.scatter(
                df_mode[width_column],
                df_mode["global_coverage"],
                s=350,
                label=f"{taxonomy_mode}-{interval_mode}",
            )

            for _, row in df_mode.iterrows():
                plt.text(
                    row[width_column],
                    row["global_coverage"],
                    str(row["n_bins"]),
                    fontsize=14,
                    ha="center",
                    va="center",
                    color="white",
                )

    plt.axhline(confidence_level, color="black", linestyle="--", linewidth=1)
    plt.xlabel(width_column)
    plt.ylabel("Global coverage")
    plt.title(f"Coverage-efficiency tradeoff ({plot_labels[label]})", fontsize=18)
    plt.grid(alpha=0.25)
    plt.legend(fontsize=14)
    plt.show()

In [ ]:
for label in label_names:
    plot_coverage_width_tradeoff(
        summary_df=candidate_df,
        label=label,
        taxonomy_modes=taxonomy_modes,
        interval_modes=interval_modes,
        confidence_level=confidence_level,
        plot_labels=plot_labels,
        width_column="global_median_width_phys",
    )

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def plot_coverage_and_width_per_bin(
    all_results,
    key,
    label_idx,
    label_names,
    plot_labels,
    confidence_level,
    width_stat="mean",          # "mean" or "median"
    width_space="std",          # "std" or "phys"
    y_std=None,                 # required if width_space="phys"
    coverage_ylim=(0.85, 0.95),
    width_ylim=None,
    annotate_counts=True,
):
    result = all_results[key]
    metrics = result.metrics

    label = label_names[label_idx]
    title_label = plot_labels[label] if label in plot_labels else label

    coverage_per_bin = metrics["coverage_per_bin"][:, label_idx]
    counts_per_bin = metrics["counts_per_bin"][:, label_idx]
    bin_tol = metrics["bin_tolerance_normal"]

    n_bins = len(coverage_per_bin)
    x = np.arange(n_bins)

    # ------------------------------------------------------------------
    # Width per bin
    # ------------------------------------------------------------------
    width_key = f"{width_stat}_width_per_bin"

    if width_key in metrics:
        width_per_bin = metrics[width_key][:, label_idx].astype(float)
    else:
        # Fallback: compute from test intervals
        bin_indices_test = result.bin_indices_test
        widths = result.upper - result.lower  # (n_test, n_labels)

        width_per_bin = np.full(n_bins, np.nan)

        for b in range(n_bins):
            mask = bin_indices_test[:, label_idx] == b

            if np.sum(mask) == 0:
                continue

            if width_stat == "mean":
                width_per_bin[b] = np.mean(widths[mask, label_idx])
            elif width_stat == "median":
                width_per_bin[b] = np.median(widths[mask, label_idx])
            else:
                raise ValueError("width_stat must be 'mean' or 'median'.")

    if width_stat not in ["mean", "median"]:
        raise ValueError("width_stat must be 'mean' or 'median'.")

    if width_space == "phys":
        if y_std is None:
            raise ValueError("y_std must be provided when width_space='phys'.")

        width_per_bin = width_per_bin * y_std[label_idx]
        width_ylabel = f"{width_stat.capitalize()} interval width [{label}]"

    elif width_space == "std":
        width_ylabel = f"{width_stat.capitalize()} interval width [standardized]"

    else:
        raise ValueError("width_space must be 'std' or 'phys'.")

    # ------------------------------------------------------------------
    # Coverage bands
    # ------------------------------------------------------------------
    low_1 = bin_tol["1sigma_low"][:, label_idx]
    low_2 = bin_tol["2sigma_low"][:, label_idx]
    low_3 = bin_tol["3sigma_low"][:, label_idx]

    high_1 = bin_tol["1sigma_high"][:, label_idx]
    high_2 = bin_tol["2sigma_high"][:, label_idx]
    high_3 = bin_tol["3sigma_high"][:, label_idx]

    # ------------------------------------------------------------------
    # Plot
    # ------------------------------------------------------------------
    fig, ax1 = plt.subplots(figsize=(9.5, 5.2))

    band_color = "tab:red"

    ax1.fill_between(
        x, low_3, high_3,
        color=band_color,
        alpha=0.10,
        label=r"Nominal 3$\sigma$",
        zorder=1,
    )

    ax1.fill_between(
        x, low_2, high_2,
        color=band_color,
        alpha=0.15,
        label=r"Nominal 2$\sigma$",
        zorder=2,
    )

    ax1.fill_between(
        x, low_1, high_1,
        color=band_color,
        alpha=0.20,
        label=r"Nominal 1$\sigma$",
        zorder=3,
    )

    ax1.plot(
        x,
        coverage_per_bin,
        marker="o",
        color="maroon",
        label="Empirical coverage",
        zorder=4,
    )

    ax1.axhline(
        confidence_level,
        linestyle="--",
        alpha=0.6,
        linewidth=1.0,
        color="black",
        label=rf"C.L. = {confidence_level}",
    )

    if annotate_counts:
        for i, n in enumerate(counts_per_bin):
            if np.isfinite(coverage_per_bin[i]):
                ax1.text(
                    i,
                    coverage_per_bin[i] + 0.005,
                    str(int(n)),
                    ha="center",
                    fontsize=9,
                    color="black",
                )

    ax1.set_xlabel("Bin index")
    ax1.set_ylabel("Coverage")
    ax1.set_ylim(*coverage_ylim)
    ax1.grid(alpha=0.25)

    # ------------------------------------------------------------------
    # Right axis: interval width
    # ------------------------------------------------------------------
    ax2 = ax1.twinx()

    ax2.plot(
        x,
        width_per_bin,
        marker="s",
        linestyle="-",
        color="tab:blue",
        label=f"{width_stat} interval width",
        zorder=5,
    )

    ax2.set_ylabel(width_ylabel)

    if width_ylim is not None:
        ax2.set_ylim(*width_ylim)

    ax1.set_title(
        f"Coverage and interval width per bin | {title_label} | {key}"
    )

    # Combine legends
    handles1, labels1 = ax1.get_legend_handles_labels()
    handles2, labels2 = ax2.get_legend_handles_labels()

    ax1.legend(
        handles1 + handles2,
        labels1 + labels2,
        ncols=2,
        loc="lower right",
    )

    plt.tight_layout()
    plt.show()

    return width_per_bin

In [ ]:
key = ("difficulty", "asymmetric", 12)

width_per_bin_phys = plot_coverage_and_width_per_bin(
    all_results=all_results,
    key=key,
    label_idx=0,
    label_names=label_names,
    plot_labels=plot_labels,
    confidence_level=0.90,
    width_stat="median",
    width_space="phys",
    y_std=y_std,
)
    

## Final candidate configurations under local-validity criteria

In [ ]:

# ============================================================
# Final best config per label under the local-validity ranking
# ============================================================

best_by_label = (
    ranking_df
    .groupby("label", group_keys=False)
    .head(1)
    .reset_index(drop=True)
)

final_cols = [
    "label",
    "taxonomy_mode",
    "interval_mode",
    "n_bins",
    "global_coverage",
    global_col,
    "min_bin_coverage",
    "max_undercoverage_gap",
    n_under_col,
    under_frac_col,
    beyond_col,
    "global_median_width_std",
    "global_median_width_phys",
    "normalized_median_width_range",
    "rel_width_excess",
    "global_lower_miss_rate",
    "global_upper_miss_rate",
    "global_tail_miss_imbalance",
    "min_count_per_bin",
]

best_by_label[final_cols]


In [ ]:
for row in best_by_label.itertuples():
    key = (row.taxonomy_mode, row.interval_mode, row.n_bins)
    label_idx = int(row.label_index)
    label = label_names[label_idx]

    print("=" * 80)
    print(f"Best configuration for label: {label}")
    print(f"key = {key}")
    print("=" * 80)

    inspect_configuration(
        all_results=all_results,
        key=key,
        label_idx=label_idx,
        label_names=label_names,
    )

    width_per_bin_phys = plot_coverage_and_width_per_bin(
        all_results=all_results,
        key=key,
        label_idx=label_idx,
        label_names=label_names,
        plot_labels=plot_labels,
        confidence_level=0.90,
        width_stat="mean",
        width_space="phys",
        y_std=y_std,
    )

In [ ]:
key = (row.taxonomy_mode, row.interval_mode, row.n_bins)
label_idx = int(row.label_index)

result = all_results[key]
metrics = result.metrics

print("key:", key)
print("label:", label_names[label_idx])

print("coverage_per_bin:")
print(metrics["coverage_per_bin"][:, label_idx])

print("counts_per_bin:")
print(metrics["counts_per_bin"][:, label_idx])

print("mean_width_per_bin:")
print(metrics["mean_width_per_bin"][:, label_idx])

print("median_width_per_bin:")
print(metrics["median_width_per_bin"][:, label_idx])

In [ ]:
mean_w = metrics["mean_width_per_bin"][:, label_idx]
median_w = metrics["median_width_per_bin"][:, label_idx]

print("mean == median per bin?")
print(np.allclose(mean_w, median_w, equal_nan=True))

print("width changes across bins?")
print(np.nanmin(mean_w), np.nanmax(mean_w))
print("unique rounded widths:")
print(np.unique(np.round(mean_w, 6)))